# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [2]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [10]:
base = pd.read_csv("Cientista de dados M35 - smoke_detection_iot.csv", delimiter =',')
base.head(4)

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0


In [12]:
base.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

In [17]:
#Para a coluna Fire Alarm, por conta do espaçamento talvez seja util renomear o nome da coluna utilizando:

base.rename(columns={'Fire Alarm': 'Fire_Alarm','Raw H2':'Raw_H2','Raw Ethanol':'Raw_Ethanol'}, inplace=True)

In [18]:
base.head(4)

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw_H2,Raw_Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire_Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0


In [19]:
base.dtypes

Unnamed: 0          int64
UTC                 int64
Temperature[C]    float64
Humidity[%]       float64
TVOC[ppb]           int64
eCO2[ppm]           int64
Raw_H2              int64
Raw_Ethanol         int64
Pressure[hPa]     float64
PM1.0             float64
PM2.5             float64
NC0.5             float64
NC1.0             float64
NC2.5             float64
CNT                 int64
Fire_Alarm          int64
dtype: object

In [23]:
base.isnull().sum()

Unnamed: 0        0
UTC               0
Temperature[C]    0
Humidity[%]       0
TVOC[ppb]         0
eCO2[ppm]         0
Raw_H2            0
Raw_Ethanol       0
Pressure[hPa]     0
PM1.0             0
PM2.5             0
NC0.5             0
NC1.0             0
NC2.5             0
CNT               0
Fire_Alarm        0
dtype: int64

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.


# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.
O modelo utilizado foi a regressão logística por se tratar de uma variável com duas categorias, é mais fácil de interpretar e simples de aplicar.

In [33]:
X=base.drop('Fire_Alarm', axis=1)
y=base['Fire_Alarm']
modelo = LogisticRegression(max_iter = 1000)
print("tamanho X", X.shape)
print("tamanho Y", y.shape)


tamanho X (62630, 15)
tamanho Y (62630,)


# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [29]:
folds = 5
crossvalidation = KFold(n_splits = folds, shuffle = True, random_state=5)

# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [31]:
modelo_final = cross_val_score(modelo, X, y, cv=folds)
pontuacoes = cross_val_score(modelo, X, y, cv = crossvalidation)
print(f"Pontuações for Fold:{pontuacoes}")
print(f"Média: {pontuacoes.mean():.4f}")

Pontuações for Fold:[0.98778541 0.9875459  0.97940284 0.97501198 0.98914258]
Média: 0.9838


O modelo apresntou um acurácia excelente tanto individualmente em cada um dos fold quato na média mesmo sem nenhum "pre processamento dos dados" tendo 98% de previsoes corretas em relação ao total feito.


Amanda Carvalheira Ribeiro 
Oi, Regina! Tudo bem?

Sou a tutora Amanda e corrigirei sua tarefa.

Parabéns pelo empenho e dedicação ao longo do curso!

Você conseguiu escolher corretamente o modelo de Machine Learning de Regressão Logística e apresentou uma justificativa coerente para sua utilização. Além disso, realizou a separação das bases de treino e da variável alvo de forma adequada, definiu o número de folds para a validação cruzada e avaliou o desempenho dos modelos por meio das métricas de pontuação, demonstrando uma boa condução da etapa de modelagem e finalizou com os seus insights sobre a avaliação dos resultados do modelo.

Você pode criar um README para o capítulo do módulo explicando o que aprendeu e quais exercícios usou para fixar o conhecimento.

Continue assim, você irá colher muitos frutos pela sua dedicação e empenho.